# E-books Market Analysis - SQL 

## Introduction and Objective

As a result of the COVID-19 pandemic, people’s habits changed significantly. By spending more time at home, many individuals adopted new activities, including reading. This created an opportunity for companies developing digital products, such as book apps. With the increase in readers, this type of application gained relevance, making the market more dynamic and competitive.

With the intention of designing a new digital product for this audience, a study will be conducted using the database of a company that competes in this market. The goal is to understand readers’ behavior and preferences and, based on these findings, formulate a strong value proposition.

### Study Objective:

To explore the catalog and user behavior, using book ratings and reviews as the primary source of analysis, in order to develop a data-driven value proposition.

All queries will be executed using SQL, and the results will be presented in this notebook.


## Database Connection

- Importing libraries

- Creating the engine

- Connection test

In [1]:
#importing driver for database connection

!pip install psycopg2-binary 

In [2]:
# Importing libraries

import pandas as pd 
from sqlalchemy import create_engine

# Creating the engine

db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [3]:
# Connection test

pd.read_sql('SELECT 1 AS test_connection;',con=engine)

,test_connection
0,1


## Defining a function to execute SQL queries

In [4]:
## Defining the function 

def run_query (query):
    return pd.read_sql(query,con=engine)

## Table Exploration

The following exploratory queries will be executed to:

- Assess table size (row counts).
- Review structure and column types.
- Detect null values and keys.
- Perform an initial preview of the available data tables.


# Data Structure Diagram 

![Data Structure Diagram](images/data_structure.png)


### Observation: Data Structure

Since the project guidelines provided the table structures and data types for each column, further exploration of data types was not required.

### Table Size Exploration

In [5]:
# Querying Table Sizes

q_table_sizes = (""" 

SELECT 'books' AS table_name, COUNT(*) AS rows FROM books
UNION ALL 
SELECT 'authors' AS table_name, COUNT(*) AS rows FROM authors
UNION ALL 
SELECT 'publishers' AS table_name, COUNT(*) AS rows FROM publishers
UNION ALL 
SELECT 'ratings' AS table_name, COUNT(*) AS rows FROM ratings
UNION ALL 
SELECT 'reviews' AS table_name, COUNT(*) AS rows FROM reviews
""")

run_query(q_table_sizes)

,table_name,rows
0,books,1000
1,authors,636
2,publishers,340
3,ratings,6456
4,reviews,2793


### Observations

Obtaining table sizes allows for a better understanding of the data volume. This information is crucial for planning the database relationships, selecting efficient JOINs, and validating the total number of records across all subsequent analyses.


### Books Table

In [6]:
# Data Preview Query

q_visual_t1 = """ SELECT * FROM books LIMIT 10 """

run_query(q_visual_t1)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268
5,6,257,1st to Die (Women's Murder Club #1),424,2005-05-20,116
6,7,258,2nd Chance (Women's Murder Club #2),400,2005-05-20,116
7,8,260,4th of July (Women's Murder Club #4),448,2006-06-01,318
8,9,563,A Beautiful Mind,461,2002-02-04,104
9,10,445,A Bend in the Road,341,2005-04-01,116


####  Identifying Nulls in Books Table


In [7]:
#  Null Value Check

q_nulls_in_books = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN book_id IS NULL THEN 1 ELSE 0 END) AS book_id_nulls,
SUM(CASE WHEN author_id IS NULL THEN 1 ELSE 0 END) AS author_id_nulls,
SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS title_nulls,
SUM(CASE WHEN num_pages IS NULL THEN 1 ELSE 0 END) AS num_pages_nulls,
SUM(CASE WHEN publication_date IS NULL THEN 1 ELSE 0 END) AS publication_date_nulls,
SUM(CASE WHEN publisher_id IS NULL THEN 1 ELSE 0 END) AS publisher_id_nulls
FROM books;
""")

run_query(q_nulls_in_books)

,total_rows,book_id_nulls,author_id_nulls,title_nulls,num_pages_nulls,publication_date_nulls,publisher_id_nulls
0,1000,0,0,0,0,0,0


### Observations: Books Table

- Initial Preview: The initial preview provides a better understanding of the data structure and column alignment.
- Null Value Check: A thorough review of null values was conducted, focusing on key columns for the analysis.
- Data Quality Results: The analysis confirms there are no null values in any of the evaluated columns. This indicates the table is complete, ensuring there are no issues that could affect JOINs or future calculations.

### Authors Table

In [8]:
# Querying the authors
q_exp_t2 = """ SELECT * FROM authors LIMIT 10 """

run_query(q_exp_t2)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd
5,6,Alan Paton
6,7,Albert Camus/Justin O'Brien
7,8,Aldous Huxley
8,9,Aldous Huxley/Christopher Hitchens
9,10,Aleksandr Solzhenitsyn/H.T. Willetts


####  Identifying Nulls in Authors Table


In [9]:
#  Null Value Check


q_nulls_in_authors = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN author_id IS NULL THEN 1 ELSE 0 END) AS author_id_nulls,
SUM(CASE WHEN author IS NULL THEN 1 ELSE 0 END) AS author_nulls
FROM authors;
""")

run_query(q_nulls_in_authors)

,total_rows,author_id_nulls,author_nulls
0,636,0,0


### Observations: Authors Table

- The Authors table contains a total of 636 records, corresponding to all authors in the catalog.
- No null values were identified in either the author_id or author_name columns, indicating the dataset is complete.
- This ensures that JOIN operations with the Books table can be performed reliably, with no risk of data loss.

### Publishers Table 

In [10]:
# Data Preview Query

q_exp_t3 = """ SELECT * FROM publishers LIMIT 10 """

run_query(q_exp_t3)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company
5,6,Aladdin
6,7,Aladdin Paperbacks
7,8,Albin Michel
8,9,Alfred A. Knopf
9,10,Alfred A. Knopf Books for Young Readers


In [11]:
#  Null Value Check

q_nulls_in_publishers = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN publisher_id IS NULL THEN 1 ELSE 0 END) AS publisher_id_nulls,
SUM(CASE WHEN publisher IS NULL THEN 1 ELSE 0 END) AS publisher_nulls
FROM publishers;
""")

run_query(q_nulls_in_publishers)

,total_rows,publisher_id_nulls,publisher_nulls
0,340,0,0


### Observations: Publishers Table

- The Publishers table contains 340 records, with no null values found in either publisher_id or the publisher's name.
- The absence of missing data guarantees a proper link with the Books table, ensuring that any publisher-level analysis can be performed accurately.

### Ratings Table

In [12]:
# Data Preview Query

q_exp_t4 = """ SELECT * FROM ratings LIMIT 10 """

run_query(q_exp_t4)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2
5,6,3,johnsonamanda,4
6,7,3,scotttamara,5
7,8,3,lesliegibbs,5
8,9,4,abbottjames,5
9,10,4,valenciaanne,4


In [13]:
#  Null Value Check

q_nulls_in_ratings = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN rating_id IS NULL THEN 1 ELSE 0 END) AS rating_id_nulls,
SUM(CASE WHEN book_id IS NULL THEN 1 ELSE 0 END) AS book_id_nulls,
SUM(CASE WHEN username IS NULL THEN 1 ELSE 0 END) AS username_nulls,
SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS rating_nulls
FROM ratings;
""")

run_query(q_nulls_in_ratings)

,total_rows,rating_id_nulls,book_id_nulls,username_nulls,rating_nulls
0,6456,0,0,0,0


### Observations: Ratings Table

- The Ratings table contains 6,456 records, reflecting high user engagement and rating activity.
- No null values were found in key columns (rating_id, book_id, username, and rating), ensuring every rating is correctly associated with both a book and a user.
- This clean structure allows for the calculation of aggregate metrics, such as averages and counts, without requiring additional data cleaning or handling of missing values.

### Reviews table

In [14]:
# Data Preview Query

q_exp_t5 = """ SELECT * FROM reviews LIMIT 10 """

run_query(q_exp_t5)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...
5,6,3,lesliegibbs,Analysis no several cause international.
6,7,4,valenciaanne,One there cost another. Say type save. With pe...
7,8,4,abbottjames,Within enough mother. There at system full rec...
8,9,5,npowers,Thank now focus realize economy focus fly. Ite...
9,10,5,staylor,Game push lot reduce where remember. Including...


In [15]:
#  Null Value Check

q_nulls_in_reviews = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS review_id_nulls,
SUM(CASE WHEN book_id IS NULL THEN 1 ELSE 0 END) AS book_id_nulls,
SUM(CASE WHEN username IS NULL THEN 1 ELSE 0 END) AS username_nulls,
SUM(CASE WHEN text IS NULL THEN 1 ELSE 0 END) AS text_nulls
FROM reviews;
""")

run_query(q_nulls_in_reviews)

,total_rows,review_id_nulls,book_id_nulls,username_nulls,text_nulls
0,2793,0,0,0,0


### Observations - Reviews Table

- The reviews table contains a total of 2,793 records and has no null values in the evaluated columns (review_id, book_id, username, and text).
- This indicates that all reviews are complete and correctly associated with both books and users.
- A lower number of records was detected compared to the ratings table; this suggests that not all users who rate a book leave a written review.

## Conclusion Initial EDA

The initial exploration confirms high data quality with zero missing values. The dataset’s integrity and structured nature allow for seamless table integration and reliable insights. This clean state eliminates the need for extensive pre-processing, enabling a direct transition to the core analytical tasks.

## User Behavior Analisis


The following SQL queries are designed to address the project's key business questions and analyze user preferences and engagement patterns.


## Number of books published after January 1, 2000


In [16]:
# Number of books published after January 1, 2000

q_task1 = ("""
SELECT 
COUNT(DISTINCT book_id)
FROM books
WHERE publication_date > '2000-01-01';
""")

run_query(q_task1)

,count
0,819


# Conclusion

Out of the 1,000 books in the catalog, 819 were published after January 1, 2000. This indicates that approximately 80% of the available titles are recent publications, reflecting a catalog primarily focused on contemporary works.

## Number of User Reviews and Average Rating per Book



In [17]:
 # Review Counts and Average Ratings by Title

q_task2 = ("""

WITH reviews_per_book AS (
SELECT 
book_id,
COUNT(review_id) AS review_count
FROM reviews
GROUP BY book_id
),

ratings_per_book AS (
SELECT
book_id,
AVG(rating) AS avg_rating
FROM ratings 
GROUP BY book_id
)
SELECT 
b.book_id,
b.title,
rpb.review_count,
rapb.avg_rating
FROM books b
LEFT JOIN reviews_per_book rpb ON b.book_id = rpb.book_id
LEFT JOIN ratings_per_book rapb ON b.book_id = rapb.book_id;

""")

run_query(q_task2)

,book_id,title,review_count,avg_rating
0,652,The Body in the Library (Miss Marple #3),2.0,4.500000
1,273,Galápagos,2.0,4.500000
2,51,A Tree Grows in Brooklyn,5.0,4.250000
3,951,Undaunted Courage: The Pioneering First Missio...,2.0,4.000000
4,839,The Prophet,4.0,4.285714
...,...,...,...,...
995,64,Alice in Wonderland,4.0,4.230769
996,55,A Woman of Substance (Emma Harte Saga #1),2.0,5.000000
997,148,Christine,3.0,3.428571
998,790,The Magicians' Guild (Black Magician Trilogy #1),2.0,3.500000


# Conclusion

The resulting dataset includes all 1,000 books, now enriched with total reviews and mean ratings. These metrics are essential for evaluating user interaction levels and the general perception of the titles available on the platform.

## Publisher with the highest number of books with more than 50 pages


In [18]:
# Top publisher by book count (books > 50 pages)

q_task3 = ("""
SELECT 
p.publisher_id,
p.publisher,
COUNT(b.book_id) AS book_count
FROM books b 
JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY
p.publisher_id,
p.publisher
ORDER BY book_count DESC
LIMIT 1;
""")

run_query(q_task3)

,publisher_id,publisher,book_count
0,212,Penguin Books,42


## Conclusion: Leading Publisher
Penguin Books was identified as the publisher with the highest number of titles exceeding 50 pages. With a total of 42 books meeting this criteria, the data suggests a strong presence in the full-length book segment, excluding brochures or short publications.
This insight is valuable for proposing strategic partnerships or content strategies with high-volume publishers. Leveraging these established relationships can enhance the user experience by offering more comprehensive and high-interest material to the platform's readers.

## Author with the highest average book rating

In [19]:
# Top-rated author by average book score

q_task4 = ("""

WITH rating_per_book AS (
SELECT 
book_id,
AVG(rating) AS avg_rating,
COUNT(rating_id) AS rating_count
FROM ratings
GROUP BY book_id
HAVING COUNT(rating_id) >= 50 
)

SELECT
a.author,
a.author_id,
AVG(rpb.avg_rating) AS author_avg_rating,
COUNT(DISTINCT b.book_id) AS books_considered
FROM rating_per_book rpb
JOIN books b ON b.book_id = rpb.book_id
JOIN authors a ON a.author_id = b.author_id
GROUP BY a.author,
a.author_id
ORDER BY author_avg_rating DESC
LIMIT 1;
""")

run_query(q_task4)

,author,author_id,author_avg_rating,books_considered
0,J.K. Rowling/Mary GrandPré,236,4.283844,4


## Conclusion: Top-Rated Author
- J.K. Rowling/Mary GrandPré was identified as the author with the highest average rating, considering only books with at least 50 reviews.
- The author achieved an average rating of 4.28 across 4 titles that met the criteria. This indicates strong user satisfaction for books with a significant volume of feedback.
 
- Strategic Insight: Prioritizing content from this author can drive higher user engagement and enhance the overall reading experience, as these recommendations are backed by broad community consensus.

## Average number of text reviews among users who rated more than 50 books

In [20]:
# Mean text reviews for high-activity users (>50 ratings)

q_task5 = ("""
WITH active_users AS (
    SELECT
        username,
        COUNT(DISTINCT book_id) AS books_rated
    FROM ratings
    GROUP BY username
    HAVING COUNT(DISTINCT book_id) > 50
),
reviews_per_user AS (
    SELECT
        username,
        COUNT(review_id) AS reviews_count
    FROM reviews
    GROUP BY username
)
SELECT
    AVG(COALESCE(rpu.reviews_count, 0)) AS avg_reviews_per_user
FROM active_users au
LEFT JOIN reviews_per_user rpu ON au.username = rpu.username;
""")

run_query(q_task5)

,avg_reviews_per_user
0,24.333333


## Final Conclusions & Value Proposition

### Conclusions

The analysis of the catalog and user behavior identified key patterns for data-driven business decision-making:

 - The vast majority of the catalog consists of contemporary titles; 819 out of 1,000 books were published after the year 2000, ensuring the offering aligns with current market interests.
- User interaction data highlights high-value publishers. Penguin Books leads the "full-length" segment (books > 50 pages) with 42 titles, positioning them as a primary strategic partner for content expansion.
- J.K. Rowling/Mary GrandPré achieved the highest average rating (4.28) among books with a significant volume of engagement (50+ ratings). This confirms a solid and reliable user preference for their work.
- While the most active users rate a high volume of books, their average number of text reviews remains relatively low. This indicates an opportunity to shift from quantitative to qualitative participation. 

### Value Proposition

To enhance the platform's market position, the following actions are recommended:

1. Optimization of Recommendation Engines: Prioritize authors and titles with both high ratings and high engagement volumes to ensure reliable user satisfaction.
2. Strategic Alliances: Formalize partnerships with top-performing publishers and authors to secure premium content.
3. Incentivizing Qualitative Feedback: Implement features or rewards to encourage written reviews, transforming simple ratings into rich, actionable user sentiment data.
   
These initiatives will improve the user experience, boost long-term engagement, and maximize the overall product value.